In [3]:
# Configuracion
N_ITER = 8
CV = 2
RANDOM_STATE = 42
TARGET_CLASS = None  # None => clase minima de y_train
BASE_NAME = None     # None => tomar ganador del resumen previo
BALANCE_NAME = None  # NONE/SMOTE/UNDER/SMOTEENN


In [4]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, make_scorer, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

REPO_ROOT = Path('.').resolve()
INPUT_BASE = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion'
PRE_REPORTS_BASE = REPO_ROOT / 'reports' / '09_modelo_lightgbm'
OUTPUT_BASE = REPO_ROOT / 'reports' / '11_tuning_lightgbm'

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'UNDER': RandomUnderSampler(random_state=RANDOM_STATE),
    'SMOTEENN': SMOTEENN(random_state=RANDOM_STATE),
}


def latest_file(pattern_base: Path, pattern: str) -> Path:
    files = sorted(pattern_base.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No se encontro archivo con patron: {pattern_base / pattern}')
    return files[-1]


def latest_input_path() -> Path:
    candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f'No hay subdirectorios en {INPUT_BASE}')
    return candidates[-1]


def choose_best(summary_csv: Path) -> tuple[str, str]:
    df = pd.read_csv(summary_csv)
    req = {'modelo', 'balanceo', 'cv_recall_target', 'cv_macro_f1'}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f'Faltan columnas en resumen: {sorted(miss)}')
    d = df.dropna(subset=['cv_recall_target', 'cv_macro_f1'])
    best = d.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]
    return str(best['modelo']), str(best['balanceo'])


def resolve_target_class(y: pd.Series, target: int | None) -> int:
    classes = list(pd.Series(y).unique())
    if target is None:
        return int(pd.Series(y).min())
    if target in classes:
        return int(target)
    return int(classes[0])


def build_pipeline(balancer, num_classes: int) -> Pipeline:
    model = LGBMClassifier(
        boosting_type='gbdt',
        objective='multiclass' if num_classes > 2 else 'binary',
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    steps = []
    if balancer is not None:
        steps.append(('balance', balancer))
    steps.append(('model', model))
    return Pipeline(steps)


In [5]:
# Seleccion de base/balanceo y carga de datos
summary_csv = latest_file(PRE_REPORTS_BASE, '**/resumen_modelos_lightgbm.csv')
input_path = latest_input_path()

base_name, balance_name = (BASE_NAME, BALANCE_NAME) if (BASE_NAME and BALANCE_NAME) else choose_best(summary_csv)
if balance_name not in BALANCE_METHODS:
    raise ValueError(f'Balanceo no soportado: {balance_name}')

X_train = pd.read_parquet(input_path / f'X_train_{base_name}.parquet')
X_test = pd.read_parquet(input_path / f'X_test_{base_name}.parquet')
y_train = pd.read_parquet(input_path / f'y_train_{base_name}.parquet').squeeze()
y_test = pd.read_parquet(input_path / f'y_test_{base_name}.parquet').squeeze()

target_class = resolve_target_class(y_train, TARGET_CLASS)
y_min = int(y_train.min())
y_train_adj = y_train - y_min
y_test_adj = y_test - y_min
target_class_adj = target_class - y_min
num_classes = int(pd.Series(y_train_adj).nunique())

balancer = BALANCE_METHODS[balance_name]
pipeline = build_pipeline(balancer, num_classes)

scorers = {
    'recall_target': make_scorer(recall_score, labels=[target_class_adj], average='macro', zero_division=0),
    'f1_macro': 'f1_macro',
}

param_distributions = {
    'model__n_estimators': [120, 180],
    'model__learning_rate': [0.05, 0.08],
    'model__num_leaves': [31, 63],
    'model__max_depth': [-1, 8],
    'model__subsample': [0.85, 1.0],
    'model__colsample_bytree': [0.85, 1.0],
    'model__min_child_samples': [20, 40],
    'model__reg_alpha': [0.0, 0.1],
    'model__reg_lambda': [1.0, 2.0],
}

cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scorers,
    refit='recall_target',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=False,
)
search.fit(X_train, y_train_adj)

best_model = search.best_estimator_
y_pred_test = best_model.predict(X_test) + y_min
y_pred_train = best_model.predict(X_train) + y_min

run_id = datetime.today().strftime('%Y%m%d')
output_dir = OUTPUT_BASE / run_id / f"LightGBM_tuning_{datetime.today().strftime('%Y-%m-%d')}"
output_dir.mkdir(parents=True, exist_ok=True)

cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_recall_target')
cv_results.to_csv(output_dir / 'cv_results_lightgbm_tuning.csv', index=False)

summary = {
    'base_name': base_name,
    'balanceo': balance_name,
    'summary_csv_source': str(summary_csv),
    'input_path': str(input_path),
    'target_class_original': int(target_class),
    'cv_best_recall_target': float(search.best_score_),
    'cv_best_f1_macro': float(cv_results.iloc[0]['mean_test_f1_macro']),
    'test_accuracy': float(accuracy_score(y_test, y_pred_test)),
    'test_f1_macro': float(f1_score(y_test, y_pred_test, average='macro', zero_division=0)),
    'train_f1_macro': float(f1_score(y_train, y_pred_train, average='macro', zero_division=0)),
}

pd.DataFrame([summary]).to_csv(output_dir / 'resumen_tuning_lightgbm.csv', index=False)
with open(output_dir / 'best_params_lightgbm.json', 'w', encoding='utf-8') as f:
    json.dump(search.best_params_, f, indent=2)

report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
report_test.to_csv(output_dir / 'classification_report_test_lightgbm.csv')

print(json.dumps(summary, indent=2))
print(f'Salida: {output_dir}')


Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068990 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 198616, number of used features: 30
[LightGBM] [Info] Start training from score -0.791541
[LightGBM] [Info] Start training from score -1.773549
[LightGBM] [Info] Start training from score -2.295491
[LightGBM] [Info] Start training from score -2.427174
[LightGBM] [Info] Start training from score -1.670639
{
  "base_name": "MinMax_ORIGINAL_PCA30",
  "balanceo": "SMOTEENN",
  "summary_csv_source": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\reports\\09_modelo_lightgbm\\20260217\\LightGBM_10_2026-02-17\\resumen_modelos_lightgbm.csv",
  "input_path": "C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\tesis_austral_2\\data\\intermediate\\05_seleccion\\04